# Session Processing

Wavesurfer → session data (licks, rewards, treadmill, bouts).

Outputs go to `pipeline_outputs_v2/spout_behavior/` on the MICROSCOPE server.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
# Or provide an explicit list of datetime.date objects.
DATES = None  # None = all dates in session range

# Overwrite mode: "skip" (default), "prompt", or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

from stroke_orofacial.animals import load_animals
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.spout_behavior.session_processing import process_animal

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
resolver = PathResolver()
animals_config = load_animals()

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        # Generate all dates in the session range
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    print(f"\n{'='*60}")
    print(f"Processing {animal_id}: {len(dates)} dates ({dates[0]} to {dates[-1]})")
    print(f"{'='*60}")

    results = process_animal(animal_id, dates, resolver, overwrite=OVERWRITE)
    n_processed = sum(1 for r in results if r)
    n_skipped = sum(1 for r in results if not r)
    print(f"  Done: {n_processed} processed, {n_skipped} skipped/missing.")